# Buổi 3 - Notebook thực hành Detection baseline (YOLO)

Notebook này giúp bạn chạy Buổi 3 theo thứ tự:
1. Cài đặt môi trường cần thiết
2. Kiểm tra dữ liệu + split
3. Tạo `data/data.yaml`
4. Train YOLO baseline
5. Chạy inference và lưu kết quả
6. Tổng hợp checkpoint cho Buổi 4

> Gợi ý: chạy từng cell từ trên xuống để dễ debug.

In [ ]:
from pathlib import Path
import os
import sys
import json

# Chuẩn hóa project root để notebook chạy được cả khi mở từ docs/ hoặc từ root
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "docs" else cwd
os.chdir(project_root)

print("Project root:", project_root)
print("Python:", sys.version.split()[0])

## 1) Cài thư viện cần thiết

- Nếu bạn đã cài từ trước thì có thể bỏ qua cell install.
- Khuyến nghị dùng cùng môi trường với các buổi trước.

In [ ]:
import subprocess

RUN_INSTALL = False  # Đổi thành True nếu muốn cài toàn bộ requirements
AUTO_INSTALL_ULTRALYTICS = True  # Tự cài ultralytics nếu thiếu

if RUN_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

if sys.version_info >= (3, 13):
    print("Canh bao: Python >= 3.13 co the khong tuong thich on dinh voi torch/ultralytics.")
    print("Khuyen nghi dung Python 3.10 hoac 3.11 neu cai dat that bai.")

# Kiểm tra package tối thiểu
try:
    __import__("ultralytics")
    print("Package ultralytics da san sang.")
except Exception:
    print("Thieu package: ultralytics")
    if AUTO_INSTALL_ULTRALYTICS:
        print("Dang thu cai ultralytics vao dung kernel hien tai...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "ultralytics"])
        __import__("ultralytics")
        print("Da cai xong ultralytics. Neu van loi, hay Restart Kernel roi chay lai.")
    else:
        print("Hay bat AUTO_INSTALL_ULTRALYTICS=True hoac tu cai bang pip.")

## 2) Kiểm tra dữ liệu và split

Cell dưới kiểm tra nhanh các thư mục chính và các file split của Buổi 2.

In [ ]:
from collections import Counter

images_dir = Path("data/images/raw")
labels_dir = Path("data/labels/raw")
splits_dir = Path("data/splits")

required_files = [
    splits_dir / "train.txt",
    splits_dir / "val.txt",
    splits_dir / "test.txt",
]

print("images_dir exists:", images_dir.exists())
print("labels_dir exists:", labels_dir.exists())
print("splits_dir exists:", splits_dir.exists())

for f in required_files:
    print(f"{f}:", "OK" if f.exists() else "MISSING")

def read_split(path: Path):
    if not path.exists():
        return []
    return [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]

split_map = {name: read_split(splits_dir / f"{name}.txt") for name in ["train", "val", "test"]}
for k, v in split_map.items():
    print(f"{k}: {len(v)} images")

# Kiểm tra nhanh số ảnh/nhãn thiếu
missing_image = Counter()
missing_label = Counter()
for split_name, image_list in split_map.items():
    for img_path in image_list:
        p = Path(img_path)
        if not p.is_absolute():
            p = project_root / p

        if not p.exists():
            missing_image[split_name] += 1
            continue

        label_path = labels_dir / f"{p.stem}.txt"
        if not label_path.exists():
            missing_label[split_name] += 1

print("Missing images:", dict(missing_image) if missing_image else "None")
print("Missing labels:", dict(missing_label) if missing_label else "None")

## 3) Tạo file `data/data.yaml`

File này dùng cho Ultralytics YOLO khi train/eval.

In [16]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

data_yaml_path = data_dir / "data.yaml"

yaml_text = """path: .
train: data/splits/train.txt
val: data/splits/val.txt
test: data/splits/test.txt

names:
  0: license_plate
"""

data_yaml_path.write_text(yaml_text, encoding="utf-8")

print("Saved:", data_yaml_path)
print(data_yaml_path.read_text(encoding="utf-8"))

Saved: data\data.yaml
path: .
train: data/splits/train.txt
val: data/splits/val.txt
test: data/splits/test.txt

names:
  0: license_plate



## 4) Train YOLO baseline (Buổi 3)

Bạn có thể chỉnh các tham số train ở cell dưới tùy theo GPU/CPU.

In [15]:
import subprocess

# Ensure ultralytics is available in the active kernel env
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    print("Khong tim thay ultralytics. Dang cai dat tu dong...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "ultralytics"])
    from ultralytics import YOLO

# Cấu hình baseline
model_name = "yolov8n.pt"
epochs = 50
imgsz = 640
batch = 16
project = "experiments"
run_name = "yolo_buoi3_baseline"

if "data_yaml_path" not in globals() or not Path(data_yaml_path).exists():
    raise FileNotFoundError("Khong tim thay data_yaml_path. Hay chay cell tao data/data.yaml truoc.")

model = YOLO(model_name)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    project=project,
    name=run_name,
    exist_ok=True,
)

print("Train finished.")
print("Save dir:", train_results.save_dir)

Ultralytics 8.4.41  Python-3.11.9 torch-2.11.0+cpu CPU (Intel Core i7-7820HQ 2.90GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_buoi3_baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

KeyboardInterrupt: 

## 5) Xem nhanh metrics sau train

Cell dưới đọc `results.csv` để xem các chỉ số cuối cùng của run.

In [14]:
import csv

save_dir = Path(train_results.save_dir)
results_csv = save_dir / "results.csv"

metric_aliases = {
    "train/box_loss": "train_box_loss",
    "train/cls_loss": "train_cls_loss",
    "metrics/precision(B)": "precision",
    "metrics/recall(B)": "recall",
    "metrics/mAP50(B)": "mAP50",
    "metrics/mAP50-95(B)": "mAP50_95",
}

if results_csv.exists():
    with results_csv.open("r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    print("Số dòng metrics:", len(rows))

    selected_rows = []
    if rows:
        selected_indices = [0, 1, 2, max(0, len(rows) - 3), max(0, len(rows) - 2), len(rows) - 1]
        seen_indices = set()
        for idx in selected_indices:
            if idx in seen_indices:
                continue
            seen_indices.add(idx)
            row = rows[idx]
            selected_rows.append({"epoch": row.get("epoch", str(idx + 1))})
            selected_rows[-1].update({alias: row.get(col, "") for col, alias in metric_aliases.items()})

    print("\nBaseline metrics ở vài epoch đầu và cuối:")
    for row in selected_rows:
        print(row)

    if rows:
        first = selected_rows[0]
        last = selected_rows[-1]
        print("\nTóm tắt ban đầu -> cuối:")
        for key in ["train_box_loss", "train_cls_loss", "precision", "recall", "mAP50", "mAP50_95"]:
            print(f"- {key}: {first.get(key)} -> {last.get(key)}")
else:
    print("Không tìm thấy results.csv. Kiểm tra lại cell train.")

Số dòng metrics: 50

Baseline metrics ở vài epoch đầu và cuối:
{'epoch': '1', 'train_box_loss': '0.72458', 'train_cls_loss': '1.28634', 'precision': '0.9409', 'recall': '0.9562', 'mAP50': '0.97145', 'mAP50_95': '0.75854'}
{'epoch': '2', 'train_box_loss': '0.74432', 'train_cls_loss': '0.72422', 'precision': '0.96957', 'recall': '0.93804', 'mAP50': '0.98515', 'mAP50_95': '0.81246'}
{'epoch': '3', 'train_box_loss': '0.74622', 'train_cls_loss': '0.5873', 'precision': '0.96209', 'recall': '0.95043', 'mAP50': '0.98146', 'mAP50_95': '0.79291'}
{'epoch': '48', 'train_box_loss': '0.42069', 'train_cls_loss': '0.21538', 'precision': '0.98757', 'recall': '0.98474', 'mAP50': '0.99384', 'mAP50_95': '0.89942'}
{'epoch': '49', 'train_box_loss': '0.41691', 'train_cls_loss': '0.20939', 'precision': '0.98663', 'recall': '0.98507', 'mAP50': '0.99365', 'mAP50_95': '0.89578'}
{'epoch': '50', 'train_box_loss': '0.40818', 'train_cls_loss': '0.20274', 'precision': '0.98852', 'recall': '0.98515', 'mAP50': '0.99

## 6) Chạy inference với checkpoint tốt nhất

Cell dưới:
- Lấy `best.pt` của run
- Copy về `weights/yolov8_license_plate.pt`
- Chạy script `scripts/run_infer.py` với `ocr-backend dummy` để tập trung đánh giá detection baseline.

In [13]:
import shutil
import subprocess

weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)

best_pt = save_dir / "weights" / "best.pt"
target_pt = weights_dir / "yolov8_license_plate.pt"

if best_pt.exists():
    shutil.copy2(best_pt, target_pt)
    print("Copied:", best_pt, "->", target_pt)
else:
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {best_pt}")

output_json = Path("outputs/predictions_buoi3.json")
output_json.parent.mkdir(parents=True, exist_ok=True)

# Demo nhanh trong notebook: chạy vài ảnh trước, không quét toàn bộ dataset.
max_images = 20
cmd = [
    sys.executable,
    "scripts/run_infer.py",
    "--input-dir", "data/images/raw",
    "--output-json", str(output_json),
    "--detector-backend", "yolov8",
    "--detector-model", str(target_pt),
    "--ocr-backend", "dummy",
    "--max-images", str(max_images),
]

print("Running:", " ".join(cmd))
completed = subprocess.run(cmd, text=True, capture_output=True)

if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

completed.check_returncode()
print(f"Saved predictions: {output_json} (max_images={max_images})")

Copied: D:\ComputerVisionNew\runs\detect\experiments\yolo_buoi3_baseline\weights\best.pt -> weights\yolov8_license_plate.pt
Running: d:\ComputerVisionNew\.venv311\Scripts\python.exe scripts/run_infer.py --input-dir data/images/raw --output-json outputs\predictions_buoi3.json --detector-backend yolov8 --detector-model weights\yolov8_license_plate.pt --ocr-backend dummy --max-images 20
Saved predictions: outputs\predictions_buoi3.json (max_images=20)


## 7) Kiểm tra nhanh output inference

In [12]:
if output_json.exists():
    preds = json.loads(output_json.read_text(encoding="utf-8"))
    print("Num predictions:", len(preds))
    print("Sample rows:")
    for row in preds[:3]:
        print(row)
else:
    print("Chưa có output JSON. Hãy chạy cell inference trước.")

Num predictions: 20
Sample rows:
{'image_id': 'plate_0001', 'plate_text': '51H12345', 'bbox_xyxy': [112, 156, 198, 183], 'confidence': 0.5, 'latency_ms': 212.9889999923762, 'source': 'data\\images\\raw\\plate_0001.png'}
{'image_id': 'plate_0002', 'plate_text': '51H12345', 'bbox_xyxy': [186, 129, 267, 151], 'confidence': 0.5, 'latency_ms': 74.4733999890741, 'source': 'data\\images\\raw\\plate_0002.png'}
{'image_id': 'plate_0003', 'plate_text': '51H12345', 'bbox_xyxy': [98, 178, 187, 203], 'confidence': 0.5, 'latency_ms': 82.0574000099441, 'source': 'data\\images\\raw\\plate_0003.png'}


## 8) Phân tích lỗi sơ bộ detection

Cell dưới xem nhanh một vài ảnh inference, vẽ bbox dự đoán và bbox ground-truth nếu có nhãn YOLO. Mục tiêu là ghi nhận lỗi sơ bộ như: detect lệch, biển nhỏ, ảnh mờ, hoặc nhầm background.

In [11]:
import math

import cv2
import matplotlib.pyplot as plt


def yolo_label_to_xyxy(label_path: Path, image_width: int, image_height: int):
    if not label_path.exists():
        return None

    lines = [ln.strip() for ln in label_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    if not lines:
        return None

    parts = lines[0].split()
    if len(parts) < 5:
        return None

    _, x_center, y_center, width, height = map(float, parts[:5])
    x1 = int((x_center - width / 2) * image_width)
    y1 = int((y_center - height / 2) * image_height)
    x2 = int((x_center + width / 2) * image_width)
    y2 = int((y_center + height / 2) * image_height)
    return [x1, y1, x2, y2]


def bbox_iou(box_a, box_b) -> float:
    if box_a is None or box_b is None:
        return 0.0

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area
    return inter_area / union_area if union_area else 0.0


def classify_detection_error(iou: float, pred_box, image_width: int, image_height: int, blur_score: float) -> str:
    if pred_box is None:
        return "không detect"

    x1, y1, x2, y2 = pred_box
    pred_area_ratio = max(0, x2 - x1) * max(0, y2 - y1) / (image_width * image_height)

    if iou >= 0.5:
        if pred_area_ratio < 0.01:
            return "detect đúng nhưng biển nhỏ"
        if blur_score < 80:
            return "detect đúng, ảnh hơi mờ"
        return "detect ổn"
    if iou >= 0.1:
        return "detect lệch"
    return "nhầm background / sai vùng"


if "output_json" not in globals() or not output_json.exists():
    raise FileNotFoundError("Chưa có predictions. Hãy chạy cell inference trước.")

preds = json.loads(output_json.read_text(encoding="utf-8"))
review_preds = preds[: min(12, len(preds))]
if not review_preds:
    raise ValueError("File predictions đang rỗng, chưa có ảnh để phân tích lỗi.")

error_rows = []
cols = 3
rows_count = math.ceil(len(review_preds) / cols)
fig, axes = plt.subplots(rows_count, cols, figsize=(15, 4 * rows_count))
axes = list(axes.flatten()) if hasattr(axes, "flatten") else [axes]

for ax, pred in zip(axes, review_preds):
    image_path = Path(pred["source"])
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        ax.set_title(f"Không đọc được ảnh: {image_path}")
        ax.axis("off")
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_height, image_width = image_rgb.shape[:2]

    pred_box = pred.get("bbox_xyxy")
    label_path = Path("data/labels/raw") / f"{image_path.stem}.txt"
    gt_box = yolo_label_to_xyxy(label_path, image_width, image_height)
    iou = bbox_iou(pred_box, gt_box)
    blur_score = cv2.Laplacian(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
    error_type = classify_detection_error(iou, pred_box, image_width, image_height, blur_score)

    if gt_box:
        x1, y1, x2, y2 = gt_box
        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

    if pred_box:
        x1, y1, x2, y2 = map(int, pred_box)
        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (255, 0, 0), 2)

    ax.imshow(image_rgb)
    ax.set_title(f"{image_path.name}\nIoU={iou:.2f} | {error_type}")
    ax.axis("off")

    error_rows.append(
        {
            "image_id": pred.get("image_id"),
            "iou": round(iou, 3),
            "confidence": pred.get("confidence"),
            "blur_score": round(float(blur_score), 1),
            "error_type": error_type,
        }
    )

for ax in axes[len(review_preds):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Ghi chú: xanh lá = ground-truth, đỏ = dự đoán.")
print("\nBảng phân tích lỗi sơ bộ:")
for row in error_rows:
    print(row)

<Figure size 1500x1600 with 12 Axes>

Ghi chú: xanh lá = ground-truth, đỏ = dự đoán.

Bảng phân tích lỗi sơ bộ:
{'image_id': 'plate_0001', 'iou': 0.908, 'confidence': 0.5, 'blur_score': 2607.4, 'error_type': 'detect ổn'}
{'image_id': 'plate_0002', 'iou': 0.875, 'confidence': 0.5, 'blur_score': 2160.8, 'error_type': 'detect ổn'}
{'image_id': 'plate_0003', 'iou': 0.949, 'confidence': 0.5, 'blur_score': 3022.8, 'error_type': 'detect ổn'}
{'image_id': 'plate_0004', 'iou': 0.938, 'confidence': 0.5, 'blur_score': 1525.3, 'error_type': 'detect ổn'}
{'image_id': 'plate_0005', 'iou': 0.839, 'confidence': 0.5, 'blur_score': 3092.7, 'error_type': 'detect ổn'}
{'image_id': 'plate_0006', 'iou': 0.943, 'confidence': 0.5, 'blur_score': 3130.2, 'error_type': 'detect ổn'}
{'image_id': 'plate_0007', 'iou': 0.952, 'confidence': 0.5, 'blur_score': 2216.4, 'error_type': 'detect ổn'}
{'image_id': 'plate_0008', 'iou': 0.929, 'confidence': 0.5, 'blur_score': 2996.5, 'error_type': 'detect ổn'}
{'image_id': 'plate_0009', 'iou': 0.932, 'confidence':

## 9) Checklist nghiệm thu Buổi 3

- [ ] Có `data/data.yaml` hợp lệ
- [ ] Train YOLO chạy hoàn tất và có `best.pt`
- [ ] Có `outputs/predictions_buoi3.json`
- [ ] Ghi lại loss/metrics baseline ở vài epoch đầu và cuối (`train/box_loss`, `train/cls_loss`, `precision`, `recall`, `mAP50`, `mAP50-95`)
- [ ] Có phân tích lỗi sơ bộ: detect lệch, biển nhỏ, ảnh mờ, nhầm background